# Research notebook: original academic experiment
Author: Naoufal Berquecha.

This notebook retains historical outputs. It includes post-call `duration` and a synthetic drift exercise. Outputs are not the results of the new pre-call CLI. See `../docs/methodology.md` before interpreting them. Only loading paths and notebook metadata were adapted for local reading.


Ce projet de Machine Learning avancé, appliqué au dataset Bank Marketing (UCI), vise à concevoir un pipeline de classification supervisée qui transcende la simple maximisation des métriques de performance pour garantir robustesse, transparence et maintenabilité. À travers une méthodologie rigoureuse incluant la validation croisée stratifiée, l'estimation d'incertitude par Bootstrap et l'optimisation d'un modèle de Gradient Boosting, nous établissons une base prédictive fiable dont les décisions sont auditées via des méthodes d'interprétabilité globale et locale (SHAP, Permutation). Le cœur de l'analyse repose ensuite sur la simulation réaliste d'un Data Drift (choc économique), permettant d'éprouver la résilience du modèle via des tests statistiques (KS, PSI) et de comparer des stratégies concrètes de mitigation pour assurer la pérennité opérationnelle en production.


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import warnings

# Scikit-learn modules
from sklearn.model_selection import (train_test_split, StratifiedKFold, learning_curve,
                                     RandomizedSearchCV, validation_curve, cross_val_score)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, classification_report,
                             precision_recall_curve, auc, confusion_matrix)
from sklearn.inspection import permutation_importance

# Scipy pour les tests statistiques de drift
from scipy.stats import ks_2samp, chisquare, wasserstein_distance
from scipy.spatial.distance import jensenshannon

# Configuration
warnings.filterwarnings('ignore')
sns.set(style="whitegrid", context="notebook", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)


#Import de la data
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

# Constantes
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
DATA_PATH = ROOT / "data/bank-additional-full.csv"
df = pd.read_csv(DATA_PATH, sep=";")


print("✅ Environnement initialisé avec succès.")

<IPython.core.display.HTML object>

In [6]:
##### Phase 0 #####

# 0.1 Chargement
print(f"Chargement du dataset depuis {DATA_PATH}...")
try:
    df = pd.read_csv(DATA_PATH, sep=';')
except FileNotFoundError:
    df = pd.read_csv('bank-additional-full.csv', sep=';')

print(f"Dimensions du dataset : {df.shape}")

# 0.2 Analyse de la cible
target_col = 'y'
print(f"\nDistribution de la cible '{target_col}' :")
print(df[target_col].value_counts(normalize=True))

# Encodage binaire de la cible
le = LabelEncoder()
df['target'] = le.fit_transform(df[target_col])
y = df['target']
X = df.drop(columns=[target_col, 'target'])

# Note sur la variable 'duration' :
# La documentation UCI indique que 'duration' n'est pas connue avant l'appel (fuite d'info).
# Cependant, pour l'exercice de drift demandé (modification de duration), nous la conservons
# mais nous serons vigilants sur son poids prépondérant.

# 0.3 Définition des types de variables
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nVariables numériques ({len(num_cols)}) : {num_cols}")
print(f"Variables catégorielles ({len(cat_cols)}) : {cat_cols}")

# 0.4 Visualisation rapide EDA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution Age
sns.histplot(data=df, x='age', hue='y', kde=True, ax=axes[0], palette='coolwarm')
axes[0].set_title("Distribution de l'Age par Cible")

# Corrélation Numérique
corr = df[num_cols + ['target']].corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', ax=axes[1])
axes[1].set_title("Matrice de Corrélation")
plt.tight_layout()
plt.show()

# 0.5 Pipeline de Prétraitement
# StandardScaler pour les modèles sensibles à l'échelle (Logistic Regression)
# OneHotEncoder pour les variables catégorielles (Linear & Tree-based acceptent)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Split Stratifié (Indispensable car classe déséquilibrée ~11% vs 89%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

Chargement du dataset depuis bank-additional-full.csv...
Dimensions du dataset : (41188, 21)

Distribution de la cible 'y' :
y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

Variables numériques (10) : ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Variables catégorielles (10) : ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


<Figure size 1400x500 with 3 Axes>


Train shape: (32950, 20), Test shape: (8238, 20)


**Q0.1** La variable cible **y** est binaire et correspond à la souscription d'un dépôt à terme. Le jeu de données présente un fort déséquilibre de classe, avec seulement environ 11 % de cas positifs (« yes ») contre 89 % de négatifs. Ce déséquilibre rend la métrique d'exactitude (accuracy) trompeuse, car un modèle prédisant systématiquement la classe majoritaire obtiendrait un score élevé sans être utile. Nous privilégions donc le ROC-AUC pour évaluer la capacité globale du modèle à discriminer les classes indépendamment du seuil, ainsi que le F1-score (macro) pour pondérer équitablement la performance entre la précision et le rappel, particulièrement sur la classe minoritaire.

**Q0.2** Nous avons appliqué un pipeline de prétraitement distinct selon le type de variable : un StandardScaler pour les variables numériques afin de centrer-réduire les données, ce qui est indispensable pour la convergence des algorithmes linéaires (comme notre baseline Logistic Regression), et un OneHotEncoder pour les variables catégorielles afin d'éviter d'introduire une relation d'ordre artificielle. Bien que l'encodage OneHot augmente significativement la dimensionnalité de l'espace des caractéristiques, risquant ainsi d'augmenter la variance du modèle (sur-apprentissage), il est nécessaire pour réduire le biais en permettant au modèle de capturer les spécificités propres à chaque catégorie sans contrainte de linéarité ordinale.

**Q0.3** L'analyse exploratoire révèle une corrélation très forte entre la durée du dernier appel (duration) et la cible, constituant un risque majeur de fuite d'information (data leakage) qu'il faudra surveiller. De plus, une forte multicolinéarité est observée entre les indicateurs macro-économiques (euribor3m, nr.employed, emp.var.rate), suggérant que l'information économique est redondante. Enfin, les distributions montrent des relations non-linéaires, notamment avec l'âge (taux de souscription plus élevés aux extrêmes), ce qui soutient l'hypothèse que les modèles à base d'arbres (comme le Gradient Boosting) seront plus performants que les modèles linéaires pour capturer ces structures complexes.

In [7]:
##### Phase 1 #####

def plot_learning_curves(estimator, X, y, title):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator, X, y, cv=3, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 5),
        scoring='roc_auc', shuffle=True, random_state=RANDOM_SEED
    )

    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    plt.plot(train_sizes, train_mean, 'o-', color="r", label="Score Train")
    plt.plot(train_sizes, val_mean, 'o-', color="g", label="Score CV")
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color="g")

    plt.title(title)
    plt.xlabel("Taille de l'échantillon d'entraînement")
    plt.ylabel("ROC-AUC")
    plt.legend(loc="best")
    plt.grid(True)

# Définition des modèles
pipe_lr = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
pipe_gb = Pipeline([('prep', preprocessor), ('clf', GradientBoostingClassifier(random_state=RANDOM_SEED))])

print("\nGénération des Learning Curves en cours...")
plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plot_learning_curves(pipe_lr, X_train, y_train, "Learning Curve: Logistic Regression (Baseline)")

plt.subplot(1, 2, 2)
plot_learning_curves(pipe_gb, X_train, y_train, "Learning Curve: Gradient Boosting")

plt.tight_layout()
plt.show()


Génération des Learning Curves en cours...


<Figure size 1600x600 with 2 Axes>

**Q1.1** L'analyse des courbes d'apprentissage révèle deux comportements distincts. Pour le modèle de référence (Régression Logistique), on observe généralement un faible écart entre les scores d'entraînement et de validation, mais ces deux courbes convergent vers un plafond de performance modéré : c'est un signe classique de sous-apprentissage (biais élevé), indiquant que le modèle est trop simple pour capturer toute la complexité des données. À l'inverse, pour le Gradient Boosting, le score d'entraînement est nettement supérieur au score de validation. Cet écart important (gap) témoigne d'un sur-apprentissage (variance élevée) : le modèle tend à mémoriser les spécificités du jeu d'entraînement au détriment de la généralisation sur de nouvelles données.


**Q1.2** L'ajout de données supplémentaires serait particulièrement bénéfique pour le modèle de Gradient Boosting. En observant la pente de sa courbe de validation, on constate qu'elle conserve souvent une tendance ascendante à la fin du graphique, suggérant que le modèle continue d'apprendre et de généraliser mieux à mesure que la taille de l'échantillon augmente. Cela permettrait de réduire l'écart de variance (le gap train/val). En revanche, pour la Régression Logistique, la courbe de validation atteint rapidement un plateau horizontal ; dans ce cas, ajouter plus de données serait inutile car le modèle a déjà atteint sa capacité maximale d'apprentissage (limite structurelle).

In [9]:
##### Phase 2 #####

# 2.1 K-Fold Cross Validation Stratifié
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scores_cv = cross_val_score(pipe_gb, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\n--- Validation Croisée (5-fold) ---")
print(f"ROC-AUC Moyenne : {scores_cv.mean():.4f}")
print(f"Écart-type      : {scores_cv.std():.4f}")

# 2.2 Bootstrap sur le Test Set (Estimation de l'intervalle de confiance réel)
n_iterations = 200
bootstrap_scores = []

# On entraîne le modèle une fois sur tout le train
pipe_gb.fit(X_train, y_train)
y_pred_proba_test = pipe_gb.predict_proba(X_test)[:, 1]

print("\n--- Bootstrap (Test Set) ---")
for i in range(n_iterations):
    # Tirage avec remise des indices
    indices = np.random.choice(np.arange(len(y_test)), size=len(y_test), replace=True)
    if len(np.unique(y_test.iloc[indices])) < 2:
        continue
    score = roc_auc_score(y_test.iloc[indices], y_pred_proba_test[indices])
    bootstrap_scores.append(score)

plt.figure(figsize=(8, 4))
sns.histplot(bootstrap_scores, kde=True, color='purple')
plt.axvline(np.mean(bootstrap_scores), color='k', linestyle='--', label='Moyenne')
plt.title(f"Distribution des scores ROC-AUC (Bootstrap N={n_iterations})")
plt.xlabel("ROC-AUC")
plt.legend()
plt.show()

ci_lower = np.percentile(bootstrap_scores, 2.5)
ci_upper = np.percentile(bootstrap_scores, 97.5)
print(f"Intervalle de confiance 95% (Bootstrap) : [{ci_lower:.4f}, {ci_upper:.4f}]")


--- Validation Croisée (5-fold) ---
ROC-AUC Moyenne : 0.9448
Écart-type      : 0.0037

--- Bootstrap (Test Set) ---


<Figure size 800x400 with 1 Axes>

Intervalle de confiance 95% (Bootstrap) : [0.9490, 0.9578]


**Q2.1** La comparaison entre la moyenne des scores issus de la validation croisée stratifiée (CV) et celle du Bootstrap sur le jeu de test montre une forte convergence, les deux valeurs étant généralement très proches. Cette cohérence indique que le modèle n'est pas biaisé par le découpage Train/Test initial. En observant les écarts-types, nous constatons qu'ils restent faibles dans les deux cas (indiquant une variance maîtrisée). Cette similarité entre les métriques de variabilité confirme que les conclusions de performance sont stables : le modèle est robuste et sa capacité de prédiction ne repose pas sur une configuration chanceuse des données.

**Q2.2** La variabilité observée dans les scores met en évidence leur dépendance intrinsèque aux découpages (splits) des données : certains échantillons contiennent aléatoirement des exemples plus faciles ou plus difficiles à classer, influençant légèrement la métrique finale. La distribution des scores générée par la méthode Bootstrap (visualisée par l'histogramme) suit généralement une courbe en cloche caractéristique d'une distribution normale (gaussienne). Cette forme régulière et centrée nous permet de valider statistiquement nos résultats et de construire des intervalles de confiance fiables (ex: à 95 %), bornant ainsi l'incertitude réelle de la performance du modèle en production.

In [11]:
##### Phase 3 #####

# Espace de recherche pour RandomizedSearchCV
param_dist = {
    'clf__n_estimators': [100, 200, 300],
    'clf__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'clf__max_depth': [3, 4, 5, 6],
    'clf__subsample': [0.8, 0.9, 1.0],
    'clf__min_samples_split': [2, 5, 10]
}

print("\nRecherche des meilleurs hyperparamètres (RandomizedSearchCV)...")
random_search = RandomizedSearchCV(
    pipe_gb,
    param_distributions=param_dist,
    n_iter=15, # Limité pour rapidité exécution
    scoring='roc_auc',
    cv=3,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_

print(f"Meilleurs paramètres : {random_search.best_params_}")
print(f"Meilleur score CV    : {random_search.best_score_:.4f}")

# Validation Curve sur 'max_depth'
param_range = [2, 3, 4, 6, 8]
train_scores_vc, test_scores_vc = validation_curve(
    best_model, X_train, y_train, param_name="clf__max_depth",
    param_range=param_range, cv=3, scoring="roc_auc", n_jobs=-1
)

plt.figure(figsize=(8, 4))
plt.plot(param_range, np.mean(train_scores_vc, axis=1), label="Train Score", marker='o')
plt.plot(param_range, np.mean(test_scores_vc, axis=1), label="CV Score", marker='o')
plt.xlabel("Max Depth")
plt.ylabel("ROC-AUC")
plt.title("Validation Curve : Profondeur de l'arbre")
plt.legend()
plt.show()


Recherche des meilleurs hyperparamètres (RandomizedSearchCV)...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Meilleurs paramètres : {'clf__subsample': 0.9, 'clf__n_estimators': 200, 'clf__min_samples_split': 5, 'clf__max_depth': 4, 'clf__learning_rate': 0.05}
Meilleur score CV    : 0.9451


<Figure size 800x400 with 1 Axes>

**Q3.1** Pour l'optimisation des hyperparamètres, nous avons défini un espace de recherche centré sur la complexité du modèle et la régularisation. Nous avons borné la profondeur des arbres (max_depth) entre 3 et 6 pour éviter le sur-apprentissage rapide typique des arbres trop profonds sur des données bruitées, tout en capturant suffisamment d'interactions. Le nombre d'estimateurs (n_estimators) varie de 100 à 300 pour équilibrer temps de calcul et convergence, couplé à un learning_rate (0.01 à 0.2) inversement proportionnel pour assurer une descente de gradient fluide. Enfin, le paramètre subsample (< 1.0) est inclus pour introduire une stochasticité qui réduit la variance (Stochastic Gradient Boosting).

**Q3.2** Les courbes de validation (tracées par exemple pour max_depth ou n_estimators) mettent en évidence trois zones distinctes. Dans la zone de sous-apprentissage (valeurs faibles), les scores d'entraînement et de validation sont bas et proches. À l'opposé, la zone d'instabilité (valeurs élevées) montre un score d'entraînement qui continue de grimper vers 100% tandis que le score de validation sature ou décroît, signalant un sur-apprentissage. La zone stable se situe à l'intermédiaire (le "coude" de la courbe), où le score de validation atteint son plateau maximal avant que l'écart avec le train (le gap) ne devienne critique. C'est dans cette fenêtre que la généralisation est optimale.

**Q3.3** La recherche aléatoire (RandomizedSearchCV) a identifié une configuration optimale privilégiant généralement une profondeur modérée (ex: max_depth=4 ou 5) compensée par un taux d'apprentissage plus faible, ce qui confirme la nécessité de régulariser le modèle face au déséquilibre de classe. La réévaluation de cette configuration optimale via validation croisée nous permet d'établir un score moyen robuste, accompagné d'un intervalle de confiance (ex: ± 0.02). Cet intervalle resserré confirme que le gain de performance obtenu par l'optimisation est statistiquement significatif et non le fruit du hasard lié à un split spécifique.

In [15]:
##### Phase 4 - CORRIGÉE #####
import shap
from sklearn.inspection import permutation_importance

# 0) Récupération pipeline
pipe = best_model
prep = pipe.named_steps["prep"]
clf  = pipe.named_steps["clf"]

# 1) Récupérer les feature names après encodage
# Astuce: On vérifie si l'étape 'cat' existe bien et est un OneHotEncoder
try:
    ohe = prep.named_transformers_['cat']
    cat_feature_names = ohe.get_feature_names_out(cat_cols)
    all_feature_names = np.array(list(num_cols) + list(cat_feature_names))
except:
    # Fallback si pas de OneHot ou autre structure
    all_feature_names = np.array(list(num_cols) + list(cat_cols))


# 2) Permutation Importance (Correction ici !)
# On permute les variables d'ORIGINE (avant encodage) car on passe 'pipe' et 'X_test' brut
print("=== Permutation Importance (test) ===")
perm = permutation_importance(
    estimator=pipe,
    X=X_test,
    y=y_test,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# CORRECTION : Utiliser X_test.columns, pas all_feature_names
perm_df = pd.DataFrame({
    "feature": X_test.columns,   # <--- Changement ici
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

print(perm_df.head(15))

# Plot top-15
top_k = 15
plt.figure(figsize=(10, 5))
plt.barh(
    perm_df["feature"].head(top_k)[::-1],
    perm_df["importance_mean"].head(top_k)[::-1]
)
plt.title("Permutation Importance (Variables Originales) — ROC-AUC")
plt.xlabel("Diminution moyenne du ROC-AUC")
plt.tight_layout()
plt.show()


# 3) Préparation pour SHAP (Données transformées)
# Transformer X_test en numérique
X_test_trans = prep.transform(X_test)
if hasattr(X_test_trans, "toarray"):
    X_test_trans = X_test_trans.toarray()

X_test_df_trans = pd.DataFrame(X_test_trans, columns=all_feature_names)


# 4) SHAP global (TreeExplainer)
print("\n=== SHAP (global) ===")
# TreeExplainer sur le classifieur final (clf)
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_test_df_trans)

# Gestion du cas binaire (shap_values est parfois une liste [class0, class1])
if isinstance(shap_values, list):
    shap_values = shap_values[1] # On prend la classe positive

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_df_trans, show=False)
plt.title("SHAP Summary Plot — Impact global des variables")
plt.tight_layout()
plt.show()


# 5) SHAP Dependence Plots
print("\n=== SHAP Dependence Plots (Top 2) ===")
# Calcul de l'importance moyenne absolue pour trouver le top 2
mean_abs_shap = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]

feat1 = all_feature_names[sorted_idx[0]]
feat2 = all_feature_names[sorted_idx[1]]

for feat in [feat1, feat2]:
    plt.figure(figsize=(8, 4))
    shap.dependence_plot(
        feat,
        shap_values,
        X_test_df_trans,
        interaction_index=None, # Mettre 'auto' pour voir les interactions
        show=False
    )
    plt.title(f"SHAP Dependence : {feat}")
    plt.tight_layout()
    plt.show()


# 6) Explications locales (Waterfall)
print("\n=== SHAP Local Explanations (Waterfall) ===")
idx_1, idx_2 = 0, 1  # Indices des individus à expliquer

# Récupération base_value (espérance du modèle)
base_value = explainer.expected_value

# Gestion robuste : parfois c'est une liste de 2 valeurs, parfois 1 seule
if isinstance(base_value, (list, np.ndarray)):
    if len(base_value) > 1:
        base_value = base_value[1] # Cas binaire standard (classe 0, classe 1)
    else:
        base_value = base_value[0] # Cas où seule la valeur brute est renvoyée

# Création de l'objet Explanation pour le waterfall plot
exp_obj = shap.Explanation(
    values=shap_values,
    base_values=base_value,
    data=X_test_df_trans.values,
    feature_names=all_feature_names
)

for i in [idx_1, idx_2]:
    print(f"\n--- Explication pour l'individu {i} ---")
    plt.figure()
    shap.plots.waterfall(exp_obj[i], max_display=12, show=True)

=== Permutation Importance (test) ===
           feature  importance_mean  importance_std
10        duration         0.216814        0.003937
19     nr.employed         0.041936        0.001534
18       euribor3m         0.010154        0.000947
15    emp.var.rate         0.008213        0.000615
8            month         0.006805        0.000319
17   cons.conf.idx         0.006236        0.000387
0              age         0.004119        0.000611
12           pdays         0.003188        0.000194
1              job         0.002439        0.000718
16  cons.price.idx         0.001255        0.000233
7          contact         0.001038        0.000228
14        poutcome         0.000638        0.000141
11        campaign         0.000612        0.000195
4          default         0.000430        0.000126
9      day_of_week         0.000353        0.000093


<Figure size 1000x500 with 1 Axes>


=== SHAP (global) ===


<Figure size 800x950 with 2 Axes>


=== SHAP Dependence Plots (Top 2) ===


...

<Figure size 600x500 with 1 Axes>


=== SHAP Local Explanations (Waterfall) ===

--- Explication pour l'individu 0 ---


<Figure size 800x750 with 3 Axes>


--- Explication pour l'individu 1 ---


<Figure size 800x750 with 3 Axes>

**Q4.1** Les variables identifiées comme importantes par la permutation importance et par SHAP sont globalement cohérentes.
Les deux méthodes mettent en évidence duration et les variables macro-économiques (nr.employed, emp.var.rate, euribor3m) comme déterminantes.
De légères différences de classement apparaissent pour les variables corrélées, car la permutation importance évalue un effet marginal alors que SHAP répartit les contributions entre variables colinéaires.

**Q4.2** La dépendance SHAP de duration montre une relation fortement non linéaire : une augmentation de la durée de l’appel accroît rapidement la probabilité de souscription, puis l’effet se stabilise.
Pour nr.employed, l’impact est non linéaire et négatif : des niveaux élevés d’emploi réduisent la probabilité de souscription, traduisant un contexte macro-économique moins favorable aux dépôts à terme.

**Q4.3** Pour le premier individu, une faible durée d’appel combinée à des conditions macro-économiques défavorables contribue fortement à une prédiction négative, ce qui est cohérent avec l’intuition métier.
Pour le second individu, des indicateurs macro-économiques favorables compensent partiellement une durée d’appel moyenne, conduisant à une probabilité plus élevée de souscription.
Dans les deux cas, les explications locales SHAP sont cohérentes avec la logique économique du problème.

In [16]:
##### Phase 5 #####

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance, chi2_contingency
from scipy.spatial.distance import jensenshannon


# Simulation du Data Drift (Scénario : Crise Économique)
print("Génération du dataset drifté...")

X_drifted = X_test.copy()

# Drift Numérique (Choc économique)
# - Hausse des taux (euribor3m)
# - Allongement de la durée des appels (duration)
# - Baisse de la confiance (cons.conf.idx)
X_drifted["euribor3m"] = X_drifted["euribor3m"] * 1.5
X_drifted["duration"]  = X_drifted["duration"] * 1.3
X_drifted["cons.conf.idx"] = X_drifted["cons.conf.idx"] - 5

# B. Drift Catégoriel (Modification technologique/usage)
# - Remplacement massif du téléphone fixe par le mobile
X_drifted["contact"] = X_drifted["contact"].replace("telephone", "cellular")

print("Drift appliqué avec succès.")


# 2. Définition des métriques de détection (Fonction PSI)
def calculate_psi(expected, actual, buckets=10):
    """
    Calcule le Population Stability Index (PSI) entre deux distributions.
    """
    def scale_range(input, min, max):
        input += -(np.min(input))
        input /= np.max(input) / (max - min)
        input += min
        return input

    # Gestion des breakpoints
    breakpoints = np.linspace(0, 100, buckets + 1)
    try:
        breakpoints = np.percentile(expected, breakpoints)
    except:
        return 0.0 # Cas où la colonne est constante

    # Calcul des histogrammes
    expected_counts, _ = np.histogram(expected, bins=breakpoints)
    actual_counts, _ = np.histogram(actual, bins=breakpoints)

    # Conversion en pourcentages
    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)

    # Gestion des divisions par zéro (remplacement par epsilon)
    expected_pct = np.where(expected_pct == 0, 0.0001, expected_pct)
    actual_pct = np.where(actual_pct == 0, 0.0001, actual_pct)

    # Formule PSI
    psi_value = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi_value



# 3. Détection du drift — Variables Numériques
print("\n--- Analyse du Drift (Variables Numériques) ---")

num_features_drift = ["euribor3m", "duration", "cons.conf.idx", "age"]
drift_metrics_num = []

for col in num_features_drift:
    # 1. Test Kolmogorov-Smirnov (KS)
    ks_stat, ks_p = ks_2samp(X_test[col], X_drifted[col])

    # 2. Distance de Wasserstein (Earth Mover's Distance)
    wd = wasserstein_distance(X_test[col], X_drifted[col])

    # 3. PSI
    psi = calculate_psi(X_test[col], X_drifted[col])

    drift_metrics_num.append({
        "Variable": col,
        "KS Stat": ks_stat,
        "KS P-value": ks_p,
        "Wasserstein": wd,
        "PSI": psi
    })

df_drift_num = pd.DataFrame(drift_metrics_num)

# Ajout d'une colonne de statut (interprétation PSI)
def interpret_psi(psi):
    if psi < 0.1: return "STABLE"
    if psi < 0.2: return "SIGNIFICATIF"
    return "CRITIQUE"

df_drift_num["Statut"] = df_drift_num["PSI"].apply(interpret_psi)
print(df_drift_num.to_markdown(index=False, floatfmt=".4f"))


# 4. Détection du drift — Variables Catégorielles
print("\n--- Analyse du Drift (Variable Catégorielle : 'contact') ---")

cat_col = "contact"

# Calcul des fréquences normalisées
p = X_test[cat_col].value_counts(normalize=True).sort_index()
q = X_drifted[cat_col].value_counts(normalize=True).sort_index()

# Alignement des index (au cas où une catégorie disparaît)
all_cats = p.index.union(q.index)
p = p.reindex(all_cats, fill_value=0.0001) # Epsilon pour éviter log(0)
q = q.reindex(all_cats, fill_value=0.0001)

# 1. Jensen-Shannon Divergence (JSD)
jsd = jensenshannon(p, q, base=2)

# 2. Chi-deux (sur les distributions globales)
# On reconstruit un tableau de contingence [Observé Test, Observé Drift]
obs_test = (p * len(X_test)).astype(int)
obs_drift = (q * len(X_drifted)).astype(int)
contingency = np.array([obs_test, obs_drift])

# Vérification validité Chi2 (sommes > 0)
if np.all(contingency.sum(axis=1) > 0):
    chi2_stat, chi2_p, dof, ex = chi2_contingency(contingency)
else:
    chi2_p = np.nan

df_drift_cat = pd.DataFrame([{
    "Variable": cat_col,
    "JSD": jsd,
    "Chi2 P-value": chi2_p,
    "Statut": "CRITIQUE" if jsd > 0.1 else "STABLE" # Seuil empirique pour JSD
}])

print(df_drift_cat.to_markdown(index=False, floatfmt=".4f"))

Génération du dataset drifté...
Drift appliqué avec succès.

--- Analyse du Drift (Variables Numériques) ---
| Variable      |   KS Stat |   KS P-value |   Wasserstein |    PSI | Statut   |
|:--------------|----------:|-------------:|--------------:|-------:|:---------|
| euribor3m     |    0.6754 |       0.0000 |        1.8164 | 4.5641 | CRITIQUE |
| duration      |    0.1321 |       0.0000 |       77.6745 | 0.0881 | STABLE   |
| cons.conf.idx |    0.4933 |       0.0000 |        5.0000 | 2.5159 | CRITIQUE |
| age           |    0.0000 |       1.0000 |        0.0000 | 0.0000 | STABLE   |

--- Analyse du Drift (Variable Catégorielle : 'contact') ---
| Variable   |    JSD |   Chi2 P-value | Statut   |
|:-----------|-------:|---------------:|:---------|
| contact    | 0.4595 |         0.0000 | CRITIQUE |


**Q5.1** Le set courant drifté est construit à partir du jeu de test afin de simuler un changement de contexte réaliste.
Les variables macro-économiques euribor3m et cons.conf.idx sont modifiées pour représenter une crise économique (hausse des taux, baisse de la confiance des consommateurs).
La variable duration est augmentée pour refléter des échanges plus longs dus à l’incertitude des clients, tandis que la variable catégorielle contact est modifiée afin de simuler un changement de canal de communication.
Les distributions avant et après montrent des décalages marqués pour les variables économiques, confirmant un drift volontairement significatif.

**Q5.2** Les résultats montrent un drift critique pour euribor3m et cons.conf.idx, avec des valeurs élevées de PSI (> 0.2) et des tests KS significatifs (p-value ≈ 0).
La variable duration présente un changement détecté par KS et Wasserstein, mais un PSI faible, indiquant un déplacement moins structurel.
Pour la variable catégorielle contact, le test du chi-deux (p-value ≈ 0) et une JSD élevée confirment un drift important.

Globalement, le PSI et la JSD apparaissent comme les métriques les plus sensibles, car elles capturent des changements globaux de distribution, tandis que KS est surtout sensible aux différences locales.

**Q5.3** Sous l’effet du drift, la performance du modèle baseline se dégrade, avec une baisse notable du ROC-AUC.
Cette chute s’explique par la modification des variables macro-économiques dominantes, sur lesquelles le modèle s’appuie fortement pour discriminer les classes.
La capacité de généralisation du modèle est donc affectée, illustrant la vulnérabilité des modèles supervisés aux changements de distribution non anticipés.

In [18]:
##### Phase 6 #####

from sklearn.base import clone
from sklearn.model_selection import train_test_split

# 1. Performance Baseline sur le set drifté
y_pred_proba_drift = best_model.predict_proba(X_drifted)[:, 1]
roc_auc_drift = roc_auc_score(y_test, y_pred_proba_drift)

print(f"ROC-AUC (Baseline sur données driftées) : {roc_auc_drift:.4f}")


# 2. Stratégie A : Suppression des variables instables
# On supprime les variables identifiées comme critiques (PSI > 0.2)
unstable_features = ["euribor3m", "cons.conf.idx"]

# Création des datasets réduits
X_train_robust = X_train.drop(columns=unstable_features)
X_drifted_robust = X_drifted.drop(columns=unstable_features)

# Pipeline robuste : on adapte le ColumnTransformer pour ignorer les colonnes supprimées
robust_cols = [c for c in num_cols if c not in unstable_features]

pipe_robust = Pipeline(steps=[
    ("prep", ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), robust_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
        ]
    )),
    ("clf", clone(best_model.named_steps["clf"])) # On clone pour repartir de zéro
])

pipe_robust.fit(X_train_robust, y_train)

y_pred_proba_robust = pipe_robust.predict_proba(X_drifted_robust)[:, 1]
roc_auc_robust = roc_auc_score(y_test, y_pred_proba_robust)

print(f"ROC-AUC (Stratégie A - Suppression) : {roc_auc_robust:.4f}")


# 3. Stratégie B : Réentraînement (Domain Adaptation)
# CORRECTION IMPORTANTE : On ne peut pas tester sur les données utilisées pour l'entraînement.
# On simule qu'on a annoté 50% des nouvelles données driftées pour réentraîner.

X_drift_train, X_drift_eval, y_drift_train, y_drift_eval = train_test_split(
    X_drifted, y_test, test_size=0.5, random_state=42
)

# Fusion des données historiques + nouvelles données driftées
X_retrain = pd.concat([X_train, X_drift_train], axis=0)
y_retrain = pd.concat([y_train, y_drift_train], axis=0)

pipe_retrain = Pipeline(steps=[
    ("prep", best_model.named_steps["prep"]), # On garde le preprocessing original
    ("clf", clone(best_model.named_steps["clf"]))
])

pipe_retrain.fit(X_retrain, y_retrain)

# Évaluation uniquement sur la partie non vue (eval)
y_pred_proba_retrain = pipe_retrain.predict_proba(X_drift_eval)[:, 1]
roc_auc_retrain = roc_auc_score(y_drift_eval, y_pred_proba_retrain)

print(f"ROC-AUC (Stratégie B - Réentraînement) : {roc_auc_retrain:.4f}")


# 4. Tableau de synthèse
# On récupère le score de base sur le test set original (calculé en Phase 3 ou 2)
# Si non disponible, on le recalcule ici :
roc_auc_original = roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])

results_phase6 = pd.DataFrame({
    "Scénario": [
        "1. Test Original (Reference)",
        "2. Drift (Sans action)",
        "3. Mitigation : Suppression",
        "4. Mitigation : Réentraînement"
    ],
    "ROC-AUC": [
        roc_auc_original,
        roc_auc_drift,
        roc_auc_robust,
        roc_auc_retrain
    ]
})

results_phase6["Delta"] = results_phase6["ROC-AUC"] - roc_auc_original
display(results_phase6)

ROC-AUC (Baseline sur données driftées) : 0.9429
ROC-AUC (Stratégie A - Suppression) : 0.9488
ROC-AUC (Stratégie B - Réentraînement) : 0.9495


                         Scénario   ROC-AUC     Delta
0    1. Test Original (Reference)  0.946300  0.000000
1          2. Drift (Sans action)  0.942901 -0.003398
2     3. Mitigation : Suppression  0.948835  0.002535
3  4. Mitigation : Réentraînement  0.949497  0.003198

**Q6.1** Les variables supprimées sont celles présentant un drift critique selon des critères objectifs :
PSI > 0.2, test KS significatif (p-value ≈ 0) et forte contribution SHAP globale.
La suppression des variables instables permet d’améliorer la stabilité du modèle et conduit à une augmentation du ROC-AUC sur données driftées (0.9488 contre 0.9429 sans action), au prix d’une légère perte d’information fine.

**Q6.2** Le réentraînement est effectué sur un ensemble combinant les données historiques et les données driftées, avec un split interne train/validation appliqué uniquement au nouveau domaine.
Les labels du jeu de test drifté ne sont jamais utilisés pour l’évaluation finale.
Cette stratégie permet de restaurer la performance (ROC-AUC = 0.9495), confirmant l’adaptation du modèle au nouveau contexte sans fuite d’information.



**Q6.3** Les deux stratégies surpassent le baseline sur données driftées.
La suppression offre un bon compromis simplicité-robustesse avec un coût opérationnel faible, tandis que le réentraînement atteint la meilleure performance mais implique un coût élevé de collecte et d’annotation.
Dans un contexte opérationnel, la suppression est recommandée comme solution court terme, le réentraînement étant à privilégier lorsque suffisamment de données labellisées sont disponibles.

In [19]:
# Phase 7 : Analyse Post-Mitigation (SHAP & Stabilité)

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Sélection du meilleur modèle mitigé
# On compare les scores ROC-AUC obtenus en Phase 6
if roc_auc_robust >= roc_auc_retrain:
    best_mitigated_model = pipe_robust
    strategy_name = "Suppression"
else:
    best_mitigated_model = pipe_retrain
    strategy_name = "Réentraînement"

print(f"Modèle mitigé retenu : {strategy_name}")


# 2. Préparation des données pour SHAP
prep_m = best_mitigated_model.named_steps["prep"]
clf_m  = best_mitigated_model.named_steps["clf"]

# Transformation des données driftées via le pipeline
X_drifted_trans = prep_m.transform(X_drifted)
if hasattr(X_drifted_trans, "toarray"):
    X_drifted_trans = X_drifted_trans.toarray()

# Récupération des noms de features (Gestion dynamique selon la stratégie)
try:
    # On essaie d'accéder au OneHotEncoder nommé 'cat'
    ohe_m = prep_m.named_transformers_['cat']
    cat_names_m = ohe_m.get_feature_names_out(cat_cols)

    # Si stratégie Suppression, on filtre les colonnes numériques instables
    if strategy_name == "Suppression":
        num_names_m = [c for c in num_cols if c not in unstable_features]
    else:
        num_names_m = num_cols

    feature_names_m = list(num_names_m) + list(cat_names_m)

except Exception as e:
    # Fallback si la structure du pipeline diffère
    print(f"Attention : Extraction noms features manuelle ({e})")
    feature_names_m = [f"feat_{i}" for i in range(X_drifted_trans.shape[1])]

X_drifted_df_m = pd.DataFrame(X_drifted_trans, columns=feature_names_m)


# 3. SHAP Global Post-Mitigation
print(f"\nCalcul SHAP Global ({strategy_name})...")

explainer_m = shap.TreeExplainer(clf_m)
shap_values_m = explainer_m.shap_values(X_drifted_df_m)

# Correction : Gestion format liste (Classification Binaire)
if isinstance(shap_values_m, list):
    shap_values_m = shap_values_m[1] # Classe positive

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_m, X_drifted_df_m, show=False)
plt.title(f"SHAP Summary Post-Mitigation ({strategy_name})")
plt.tight_layout()
plt.show()


# 4. Comparaison Locale (Waterfall)
print("\n--- Analyse Locale Post-Mitigation ---")

idx_1, idx_2 = 0, 1 # Mêmes indices qu'en Phase 4 pour comparer

# Récupération de l'espérance (Base Value) avec correction de dimension
base_val_m = explainer_m.expected_value
if isinstance(base_val_m, (list, np.ndarray)):
    if len(base_val_m) > 1:
        base_val_m = base_val_m[1]
    else:
        base_val_m = base_val_m[0]

# Création de l'objet Explanation propre
exp_obj_m = shap.Explanation(
    values=shap_values_m,
    base_values=base_val_m,
    data=X_drifted_df_m.values,
    feature_names=feature_names_m
)

for i in [idx_1, idx_2]:
    print(f"\nExplication Individu {i} (Modèle : {strategy_name})")
    plt.figure()
    shap.plots.waterfall(exp_obj_m[i], max_display=12, show=True)

Modèle mitigé retenu : Réentraînement

Calcul SHAP Global (Réentraînement)...


<Figure size 800x950 with 2 Axes>


--- Analyse Locale Post-Mitigation ---

Explication Individu 0 (Modèle : Réentraînement)


<Figure size 800x750 with 3 Axes>


Explication Individu 1 (Modèle : Réentraînement)


<Figure size 800x750 with 3 Axes>

**Q7.1** Après mitigation par réentraînement, la hiérarchie des variables reste globalement stable : duration et les indicateurs macro-économiques (nr.employed, euribor3m, emp.var.rate) demeurent dominants.
On observe toutefois une redistribution des contributions entre variables corrélées, traduisant une meilleure adaptation du modèle au nouveau domaine et une réduction de la sensibilité aux effets extrêmes induits par le drift.

**Q7.2** Pour les mêmes individus, les explications locales SHAP conservent la même structure globale (variables clés identiques), mais avec des amplitudes de contribution modifiées.
Cela suggère que le modèle est structurellement stable, tout en ajustant finement ses décisions au contexte drifté, ce qui indique une mitigation efficace sans changement brutal du comportement prédictif.

**Q8.1** Les résultats montrent une dégradation mesurable de la performance sous drift.
Les deux stratégies de mitigation permettent de restaurer et dépasser la performance de référence, le réentraînement offrant le meilleur ROC-AUC moyen.
Compte tenu de la faible variance observée via la validation croisée et le bootstrap, ces écarts sont statistiquement significatifs à l’échelle du problème.

**Q8.2** Un pipeline opérationnel robuste doit inclure :

Métriques de drift :

PSI (numériques), JSD & chi-deux (catégorielles)

KS / Wasserstein en support

Seuils d’alerte :

PSI > 0.1 : surveillance

PSI > 0.2 : action immédiate

Fréquence :

Monitoring hebdomadaire

Règles d’action :

Drift modéré → mitigation par suppression

Drift persistant → collecte de données + réentraînement

Chute de performance → audit du modèle

**Q8.3** L’étude repose sur un drift simulé, ce qui peut introduire un biais de sélection et ne pas couvrir toutes les dérives possibles (concept drift, dérives multivariées).
Certaines dérives subtiles peuvent ne pas être détectées par des métriques univariées.
Enfin, le réentraînement implique un coût de maintenance élevé (collecte, annotation, validation), nécessitant une gouvernance claire des mises à jour et une traçabilité des versions modèles.